# Gerador de Senhas Seguras — Mini-projeto UFG

Projeto educativo em **Python**, elaborado com apoio de **IA Generativa**, com testes automatizados. O objetivo é gerar senhas aleatórias com tamanho e categorias configuráveis, utilizando o módulo `secrets` da biblioteca padrão.

**Como usar:** execute as células em ordem, usando o ▶ ou o menu **Ambiente de execução → Executar tudo**.

> Não registre nem compartilhe senhas verdadeiras em um notebook público. As senhas geradas aqui são apenas exemplos para demonstração.

## 1. Código principal

A função recebe o tamanho (entre 8 e 128) e quatro opções para incluir letras minúsculas, maiúsculas, números e símbolos. Garante ao menos um caractere de cada categoria escolhida e depois embaralha de modo criptograficamente apropriado.

In [5]:
import secrets
import string


def gerar_senha(tamanho=16, minusculas=True, maiusculas=True,
                numeros=True, simbolos=True):
    """Gera senha com pelo menos um caractere de cada categoria selecionada."""
    if type(tamanho) is not int or not 8 <= tamanho <= 128:
        raise ValueError("O tamanho deve ser um inteiro entre 8 e 128.")

    grupos = []
    if minusculas:
        grupos.append(string.ascii_lowercase)
    if maiusculas:
        grupos.append(string.ascii_uppercase)
    if numeros:
        grupos.append(string.digits)
    if simbolos:
        grupos.append(string.punctuation)

    if not grupos:
        raise ValueError("Selecione pelo menos uma categoria.")

    caracteres = [secrets.choice(grupo) for grupo in grupos]
    alfabeto = "".join(grupos)
    while len(caracteres) < tamanho:
        caracteres.append(secrets.choice(alfabeto))

    # Embaralhamento de Fisher-Yates usando aleatoriedade segura.
    for i in range(len(caracteres) - 1, 0, -1):
        j = secrets.randbelow(i + 1)
        caracteres[i], caracteres[j] = caracteres[j], caracteres[i]
    return "".join(caracteres)

## 2. Gerar três senhas de exemplo

Rode esta célula para visualizar o funcionamento inicial.

In [6]:
for i in range(3):
    print(f"Senha {i + 1}: {gerar_senha(16)}")

Senha 1: _MRj6{uPqTbIoN]|
Senha 2: a}/R3QG$8f]yS.Dr
Senha 3: Kq6F_OI%LM4_m"J%


## 3. Personalizar os parâmetros

Altere os valores na célula: `tamanho` entre 8 e 128; `True` significa incluir a categoria, `False` significa excluir.

In [7]:
tamanho = 20
minusculas = True
maiusculas = True
numeros = True
simbolos = False

senha_personalizada = gerar_senha(
    tamanho=tamanho,
    minusculas=minusculas,
    maiusculas=maiusculas,
    numeros=numeros,
    simbolos=simbolos,
)
print("Exemplo de senha personalizada:", senha_personalizada)

Exemplo de senha personalizada: lRXyTuAbP4mafSDER4vT


## 4. Testes automatizados

Os testes verificam tamanho, categorias obrigatórias, exclusão de símbolos, condições inválidas e variabilidade básica. A aleatoriedade não pode ser considerada comprovada apenas porque dois resultados diferem; esta validação exige outros métodos. Os testes aqui **não provam resistência criptográfica**, que depende do módulo `secrets` e do uso adequado da senha.

In [8]:
import unittest


class TestGeradorDeSenhas(unittest.TestCase):
    def test_tamanho_padrao(self):
        self.assertEqual(len(gerar_senha()), 16)

    def test_tamanho_customizado(self):
        self.assertEqual(len(gerar_senha(24)), 24)

    def test_tamanho_minimo(self):
        self.assertEqual(len(gerar_senha(8)), 8)

    def test_tamanho_maximo(self):
        self.assertEqual(len(gerar_senha(128)), 128)

    def test_caracteres_de_todos_os_grupos(self):
        senha = gerar_senha(20)
        for grupo in (string.ascii_lowercase, string.ascii_uppercase,
                      string.digits, string.punctuation):
            self.assertTrue(any(c in grupo for c in senha))

    def test_apenas_numeros(self):
        senha = gerar_senha(12, minusculas=False, maiusculas=False,
                            simbolos=False)
        self.assertTrue(all(c in string.digits for c in senha))

    def test_apenas_minusculas(self):
        senha = gerar_senha(12, maiusculas=False, numeros=False,
                            simbolos=False)
        self.assertTrue(all(c in string.ascii_lowercase for c in senha))

    def test_apenas_maiusculas(self):
        senha = gerar_senha(12, minusculas=False, numeros=False,
                            simbolos=False)
        self.assertTrue(all(c in string.ascii_uppercase for c in senha))

    def test_apenas_simbolos(self):
        senha = gerar_senha(12, minusculas=False, maiusculas=False,
                            numeros=False)
        self.assertTrue(all(c in string.punctuation for c in senha))

    def test_sem_simbolos(self):
        senha = gerar_senha(20, simbolos=False)
        self.assertTrue(all(c not in string.punctuation for c in senha))

    def test_tamanho_muito_curto(self):
        with self.assertRaises(ValueError):
            gerar_senha(7)

    def test_tamanho_muito_longo(self):
        with self.assertRaises(ValueError):
            gerar_senha(129)

    def test_tamanho_invalido(self):
        for valor in ('dez', 12.5, None, True):
            with self.subTest(valor=valor), self.assertRaises(ValueError):
                gerar_senha(valor)

    def test_nenhuma_categoria(self):
        with self.assertRaises(ValueError):
            gerar_senha(12, minusculas=False, maiusculas=False,
                        numeros=False, simbolos=False)

    def test_categorias_selecionadas_ao_mesmo_tempo(self):
        senha = gerar_senha(8, minusculas=False, simbolos=False)
        self.assertTrue(any(c in string.ascii_uppercase for c in senha))
        self.assertTrue(any(c in string.digits for c in senha))
        self.assertTrue(all(c in string.ascii_uppercase + string.digits for c in senha))


resultado = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestGeradorDeSenhas)
)
assert resultado.wasSuccessful(), "Há testes falhando."

test_apenas_maiusculas (__main__.TestGeradorDeSenhas.test_apenas_maiusculas) ... ok
test_apenas_minusculas (__main__.TestGeradorDeSenhas.test_apenas_minusculas) ... ok
test_apenas_numeros (__main__.TestGeradorDeSenhas.test_apenas_numeros) ... ok
test_apenas_simbolos (__main__.TestGeradorDeSenhas.test_apenas_simbolos) ... ok
test_caracteres_de_todos_os_grupos (__main__.TestGeradorDeSenhas.test_caracteres_de_todos_os_grupos) ... ok
test_categorias_selecionadas_ao_mesmo_tempo (__main__.TestGeradorDeSenhas.test_categorias_selecionadas_ao_mesmo_tempo) ... ok
test_nenhuma_categoria (__main__.TestGeradorDeSenhas.test_nenhuma_categoria) ... ok
test_sem_simbolos (__main__.TestGeradorDeSenhas.test_sem_simbolos) ... ok
test_tamanho_customizado (__main__.TestGeradorDeSenhas.test_tamanho_customizado) ... ok
test_tamanho_invalido (__main__.TestGeradorDeSenhas.test_tamanho_invalido) ... ok
test_tamanho_maximo (__main__.TestGeradorDeSenhas.test_tamanho_maximo) ... ok
test_tamanho_minimo (__main__.Test

## 5. Documentação e submissão

**Mini-projeto:** gerador de senhas seguras em Python.

**Objetivo:** gerar senhas aleatórias configuráveis, com validação de parâmetros e testes automatizados.

**IA Generativa:** utilizada como apoio para estruturar a função, propor testes e documentar o projeto. As decisões e validações devem ser revisadas pela pessoa responsável.

**Desafios:** garantir pelo menos um caractere de cada categoria, evitar uso de `random`, testar entradas inválidas e documentar a execução no Google Colab.

**Próximas etapas:**
1. Execute todas as células e confira o resultado dos testes.
2. No Google Colab, escolha **Arquivo → Salvar uma cópia no GitHub** (quando disponível), ou baixe o `.ipynb` e faça upload para seu repositório no GitHub.
3. Publique o código Python, testes e README, também disponíveis no pacote ZIP do projeto.
4. Insira o link do repositório e uma breve reflexão no fórum da Unidade 5 do Laboratório Introdutório (C4).

**Importante:** o repositório só está publicado quando você realizar a operação na sua conta GitHub. Este notebook não publica nada automaticamente.